In [ ]:
from typing import Iterable, Iterator
import regex as re
from tests.common import gpt2_bytes_to_unicode
import json

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

class Tokenizer:
    def __init__(self, vocab: dict[int, bytes], merges: list[tuple[bytes, bytes]], special_tokens: list[str] | None = None):
        self.vocab = vocab
        self.merges = merges
        
        self.special_tokens = special_tokens if special_tokens else []
        self.special_tokens_encoded = [tok.encode('UTF-8') for tok in self.special_tokens]
        ix = max(self.vocab.keys()) + 1 # get index definitely not in the vocab.
        for token in self.special_tokens_encoded:
            if token not in self.vocab.values():
                self.vocab[ix] = token
                ix += 1
        self.vtoi = {s:i for (i,s) in self.vocab.items()}
    
    @classmethod
    def from_files(cls, vocab_filepath: str, merges_filepath: str, special_tokens: list[str] | None = None):
        gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
        with open(vocab_filepath) as vocab_f:
            gpt2_vocab = json.load(vocab_f)
        gpt2_bpe_merges = []
        with open(merges_filepath) as f:
            for line in f:
                cleaned_line = line.rstrip()
                if cleaned_line and len(cleaned_line.split(" ")) == 2:
                    gpt2_bpe_merges.append(tuple(cleaned_line.split(" ")))
        # The GPT-2 tokenizer uses a remapped unicode encoding for bytes. Let's
        # just return the original bytes, so we don't force students to use
        # any particular encoding scheme.
        vocab = {
            gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
            for gpt2_vocab_item, gpt2_vocab_index in gpt2_vocab.items()
        }

        merges = [
            (
                bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
                bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
            )
            for merge_token_1, merge_token_2 in gpt2_bpe_merges
            ]
        return cls(vocab, merges, special_tokens)
    
    def _apply_merge(self, pretoken: list[bytes]) -> list[bytes]:
        ix = 0
        while ix < len(self.merges):
            jx = 0
            new_pretoken = []
            while jx < len(pretoken):
                if jx < len(pretoken) - 1 and (pretoken[jx], pretoken[jx+1]) == self.merges[ix]:
                    new_pretoken.append(pretoken[jx] + pretoken[jx+1])
                    jx += 2
                else:
                    new_pretoken.append(pretoken[jx])
                    jx += 1
            pretoken = new_pretoken
            ix += 1

        return pretoken
    
    def _pretoken_to_integer(self, pretoken: list[bytes]) -> list[int]:
        return [self.vtoi[b] for b in pretoken]
    
    def _merge_with_special_tokens(self, encoded_chunks: list[list[int]], special_matches: list[str]) -> list[int]:
        ix = 0
        final_result: list[int] = []
        while ix < len(encoded_chunks):
            final_result += encoded_chunks[ix]
            if ix < len(special_matches):
                key = special_matches[ix].encode('UTF-8')
                if key == b'':
                    ix += 1
                    continue
                final_result.append(self.vtoi[key])
            ix += 1
            
        return final_result
    
    def encode(self, text: str) -> list[int]:
        # Maybe first iterate over text and get all the special tokens in place?
        special_matches = []
        if len(self.special_tokens) == 0:
            special_chunks = [text]
        else:
            special_pattern = "|".join(re.escape(tok) for tok in sorted(self.special_tokens, key=len, reverse=True))
            special_matches = re.findall(special_pattern, text)
            special_chunks = re.split(special_pattern, text)
        
        encoded_chunks: list[list[int]] = []
        for chunk in special_chunks:
        # Let's split:
            encoded_chunk: list[int] = []
            for pretoken in re.finditer(PAT, chunk):
                pretoken = pretoken[0]
                pretoken = [bytes([b]) for b in pretoken.encode('UTF-8')]
                
                pretoken = self._apply_merge(pretoken)
                pretoken = self._pretoken_to_integer(pretoken)
                encoded_chunk += pretoken
            encoded_chunks.append(encoded_chunk)

        return self._merge_with_special_tokens(encoded_chunks, special_matches)
    
    
    def encode_iterable2(self, iterable: Iterable[str]) -> Iterator[int]:
        buffer: str = ""
        extra_length = max(self.special_tokens, key=len) + 1 # We don't look at the last extra_length characters.
        
        it = iter(iterable)
        
        try:
            while True:
                item = next(it)
                yield
        except StopIteration:
            print("done")
            
    
    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        for text in iterable:
            for token_id in self.encode(text):
                yield token_id
    
    def decode(self, ids: list[int]) -> str:
        byte_seq: bytes = b''
        for id in ids:
            if id not in self.vocab.keys():
                byte_seq += bytes([254])
            else:
                byte_seq += self.vocab[id]
        return byte_seq.decode('UTF-8', errors='replace')
    

In [299]:
from tests.test_tokenizer import get_tokenizer_from_vocab_merges_path

tokenizer = Tokenizer(vocab, merges)


In [300]:

all_ids = []
with open("tests/fixtures/tinystories_sample.txt") as f:
    print(list(tokenizer.encode_iterable(f)))


[198, 7454, 2402, 257, 640, 612, 373, 257, 1310, 2933, 3706, 3932, 13, 3932, 6151, 284, 7301, 262, 995, 1088, 683, 13, 679, 2497, 867, 4998, 1243, 11, 588, 4950, 410, 1386, 326, 547, 319, 3359, 287, 257, 3650, 13, 1881, 1110, 11, 3932, 373, 6155, 832, 262, 3650, 618, 339, 1625, 1973, 257, 845, 2041, 410, 589, 13, 1649, 3932, 2497, 340, 339, 373, 24562, 0, 198, 1544, 531, 11, 564, 250, 22017, 11, 326, 318, 257, 1107, 4998, 410, 589, 0, 1680, 314, 2822, 340, 30, 447, 251, 198, 464, 6128, 13884, 13541, 290, 531, 11, 564, 250, 5189, 1781, 345, 460, 13, 921, 460, 1011, 340, 1363, 290, 905, 477, 534, 2460, 703, 4998, 340, 318, 0, 447, 251, 198, 2396, 3932, 1718, 262, 410, 589, 1363, 290, 339, 373, 523, 6613, 286, 340, 0, 679, 1444, 465, 2460, 625, 290, 3751, 606, 262, 4998, 410, 589, 13, 1439, 465, 2460, 1807, 262, 410, 589, 373, 4950, 290, 3521, 470, 1975, 703, 9670, 3932, 373, 13, 198, 1870, 326, 338, 703, 3932, 1043, 281, 4998, 410, 589, 287, 262, 3650, 0, 198, 27, 91, 437, 1659, 5239, 91

In [ ]:
reference_tokenizer.encode(corpus_contents)

[15137]

In [296]:
vocab_path = "tests/fixtures/gpt2_vocab.json"
merges_path = "tests/fixtures/gpt2_merges.txt"

In [297]:
from tests.adapters import get_tokenizer
from tests.common import FIXTURES_PATH, gpt2_bytes_to_unicode
import json

In [298]:
gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
with open(vocab_path) as vocab_f:
    gpt2_vocab = json.load(vocab_f)
gpt2_bpe_merges = []
with open(merges_path) as f:
    for line in f:
        cleaned_line = line.rstrip()
        if cleaned_line and len(cleaned_line.split(" ")) == 2:
            gpt2_bpe_merges.append(tuple(cleaned_line.split(" ")))
# The GPT-2 tokenizer uses a remapped unicode encoding for bytes. Let's
# just return the original bytes, so we don't force students to use
# any particular encoding scheme.
vocab = {
    gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
    for gpt2_vocab_item, gpt2_vocab_index in gpt2_vocab.items()
}

merges = [
    (
        bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
        bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
    )
    for merge_token_1, merge_token_2 in gpt2_bpe_merges
]

In [ ]:
tokenizer = Tokenizer(vocab, merges)
tokenizer.encode('four')

[14337]

In [ ]:
(b'o', b'u') in merges

True